In [ ]:
stack = "TEST"
# stack = "STAGING"
# stack = "PRODUCTION"


# %load_ext autoreload
# %autoreload 2

In [ ]:
import datetime
import subprocess
import httpx
import json
import boto3
import tqdm
import os
import operator
import sys
from pathlib import Path
import pandas as pd
from google.cloud import tasks_v2
from dotenv import load_dotenv

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "cerulean_cloud").is_dir()
)
sys.path.insert(0, str(repo_root))
from cerulean_cloud.cloud_function_historical_run.inventory import (  # noqa: E402
    list_sentinel1_scene_ids,
    filter_sentinel1_scene_ids,
)

load_dotenv(".env")


(
    NAME,
    TITILER_URL,
    TITILER_API_KEY,
    API_KEY,
    INFERENCE_URL,
    DB_URL,
    ASA_QUEUE,
    ASA_URL,
    SR_URL,
    ORCHESTRATOR_URL,
) = operator.itemgetter(
    "NAME",
    "TITILER_URL",
    "TITILER_API_KEY",
    "API_KEY",
    "INFERENCE_URL",
    "DB_URL",
    "ASA_QUEUE",
    "ASA_URL",
    "SR_URL",
    "ORCHESTRATOR_URL",
)(json.loads(os.getenv("STACK_SECRETS"))[stack])

print((NAME + "\n") * 10)


def historical_run(payload):
    pulumi_stack = {"TEST": "test", "STAGING": "staging", "PRODUCTION": "prod20240903"}[
        stack
    ]
    url = os.getenv("HISTORICAL_RUN_URL")
    if not url:
        url = subprocess.check_output(
            [
                "pulumi",
                "stack",
                "output",
                "historical_run_url",
                "--stack",
                pulumi_stack,
            ],
            text=True,
        ).strip()
    response = httpx.post(
        url,
        json=payload,
        headers={"Authorization": f"Bearer {API_KEY}"},
        timeout=600,
    )
    response.raise_for_status()
    return response.json()

In [ ]:
# FROM LIST
scenes = [
    "S1A_S3_GRDH_1SDV_20230807T171941_20230807T172010_049772_05FC2A_B3DA",
]

print(len(scenes))

In [ ]:
# FROM CSV
scenes_csv = "/Users/jonathanraphael/Downloads/scene_ids_to_rerun.csv"
scenes_pd = pd.read_csv(scenes_csv)
scenes = scenes_pd["scene_id"].tolist()
print(len(scenes))

In [ ]:
# FROM UPSTREAM DATE RANGE (inclusive UTC acquisition dates)
# Local listing creates no batch or tasks and is not limited by the function timeout.
# Scene-relevancy is optional. No catalog credentials or deployed endpoint are needed.
start_date = datetime.date(2025, 1, 1)
end_date = datetime.date(2025, 1, 2)

scenes = list_sentinel1_scene_ids(
    start_date,
    end_date,
    modes=("IW", "EW"),  # None includes all modes
    polarizations=("DV", "SV"),  # None includes all polarizations
)
# Optionally load a region FeatureCollection: geometry = json.load(open("region.geojson"))
geometry = None
APPLY_RELEVANCY = False  # True applies ocean, VV, and resolution checks.
scenes = filter_sentinel1_scene_ids(
    scenes, geometry=geometry, apply_relevancy=APPLY_RELEVANCY
)
pd.DataFrame({"scene_id": scenes}).to_csv("scene_ids.csv", index=False)
print(f"{len(scenes):,} selected upstream scenes")

In [ ]:
# # FROM CSV
# scenes_csv = "/Users/jonathanraphael/Downloads/hitl_true_scenes.csv"
# scenes_pd = pd.read_csv(scenes_csv)
# processed = scenes_pd["scene_id"].tolist()
# print(len(processed))

# scenes = [s for s in scenes if s not in processed]
# print(len(scenes))

# ORCHESTRATOR (historical-run queue)

Submit the selected `scenes` through the deployed historical-run endpoint. It records a HISTORICAL_RUN batch and uses the stack's orchestrator queue and concurrency settings. Repeated IDs are submitted once. No local Cloud Tasks credentials are needed for orchestration.

`DRY_RUN = None` uses the stack's historical-run setting; set `True` or `False` to override it. `MANUAL_TRIGGER = True` gives downstream source-association tasks run-specific names. The response reports submissions, not completed processing.

The endpoint URL comes from `HISTORICAL_RUN_URL` when set, otherwise from the selected Pulumi stack. Deploy the historical-run changes before using the new list/explicit-scene requests. The upstream date-range cell defaults to `APPLY_RELEVANCY = False` to preserve unfiltered archive selection. Set it to `True` for ocean, VV, and resolution checks. An explicitly supplied region is always applied. Explicit CSV/DB IDs bypass relevance filtering. Both CLI discovery commands support geographic selection.


In [ ]:
DRY_RUN = None
MANUAL_TRIGGER = True

payload = {"scene_ids": scenes, "manual_trigger": MANUAL_TRIGGER}
if DRY_RUN is not None:
    payload["dry_run"] = DRY_RUN
print(historical_run(payload))

# ASA --SLOW--

In [ ]:
import concurrent.futures
import uuid


ASA_RUN_NAME = uuid.uuid4().hex


def create_task(scene_id, parent, run_name=ASA_RUN_NAME):
    # Define the payload for the task
    payload = {
        "scene_id": scene_id,
        "dry_run": False,
        "run_name": run_name,
        "run_flags": [],  # Empty list will backfill GSP, comment out to run all
    }

    # Define the task with an HTTP request
    task_id = f"asa-{scene_id}-{run_name}"
    task = {
        "name": client.task_path("cerulean-338116", "europe-west1", ASA_QUEUE, task_id),
        "http_request": {  # Specify the type of request
            "http_method": tasks_v2.HttpMethod.POST,
            "url": ASA_URL,  # The URL to send the request to
            "headers": {
                "Content-type": "application/json",
                "Authorization": f"Bearer {API_KEY}",
            },
            "body": json.dumps(
                payload
            ).encode(),  # Convert payload to JSON and encode to bytes
        },
    }

    try:
        # Create the task in the specified queue
        response = client.create_task(request={"parent": parent, "task": task})
        response
    except Exception as e:
        print(f"Error creating task for scene_id {scene_id}: {e}")


os.environ.pop("GOOGLE_APPLICATION_CREDENTIALS", None)
client = tasks_v2.CloudTasksClient()
parent = client.queue_path("cerulean-338116", "europe-west1", ASA_QUEUE)

max_workers = 100  # Example of aggressive concurrency
with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    results = list(
        tqdm.tqdm(
            executor.map(lambda scene_id: create_task(scene_id, parent), scenes),
            total=len(scenes),
        )
    )

# for scene_id in tqdm.tqdm(scenes):
#     create_task(scene_id, parent)

# ASA --FAST--

In [ ]:
import uuid

ASA_RUN_NAME = uuid.uuid4().hex
for scene_id in tqdm.tqdm(scenes):
    try:
        asa_result = httpx.post(
            ASA_URL,
            json={
                "scene_id": f"{scene_id}",
                "dry_run": False,
                "run_name": ASA_RUN_NAME,
                # "run_flags": ["INFRA"],
            },
            timeout=0.1,
            headers={"Authorization": f"Bearer {API_KEY}"},
        )
    except Exception:
        # print(e)
        pass

In [ ]:
[f"https://cerulean-ui-test-5qkjkyomta-uk.a.run.app/?scene_id={a}" for a in scenes]

https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Catalog/Examples.html

# AWS OpenDataRegistry ls and productInfo.json

## Download GeoTiffs from AWS

In [ ]:
# Download GeoTiffs directly from S3 Open Data Registry using request payer and a list of Scene IDs

import tqdm

s3_client = boto3.client("s3")
bucket_name = "sentinel-s1-l1c"

for scene_id in tqdm.tqdm(scenes):
    measurement = f"GRD/{scene_id[17:21]}/{int(scene_id[21:23])}/{int(scene_id[23:25])}/IW/DV/{scene_id}/measurement/"
    try:
        obj = s3_client.get_object(
            Bucket=bucket_name, Key=measurement + "iw-vv.tiff", RequestPayer="requester"
        )
        file_path = f"/Users/jonathanraphael/git/ceruleanserver/local/temp/outputs/_rasters/{scene_id}.tiff"
        with open(file_path, "wb") as file:
            chunks = []
            while data := obj["Body"].read(1024):
                chunks.append(data)
            file.write(b"".join(chunks))

        print(f"{scene_id} downloaded successfully!")
    except Exception:
        command = (
            f"aws s3 ls s3://{bucket_name}/{measurement} --request-payer requester"
        )
        output = subprocess.check_output(command, shell=True)
        print(output)
        print(f"{scene_id} is missing the VV file. Perhaps one of the above instead?")


print("wrapped")

In [ ]:
import asyncio
import httpx
import boto3
import json
import subprocess
import tqdm

start_date = datetime.date(2023, 1, 10)
end_date = datetime.date(2023, 1, 11)
concurrency = 25

s3_client = boto3.client("s3")
bucket_name = "sentinel-s1-l1c"
current_date = start_date
semaphore = asyncio.Semaphore(concurrency)


def extract_folders(output):
    lines = output.decode("utf-8").split("\n")
    folder_names = []
    for line in lines:
        if line.strip().startswith("PRE"):
            folder_name = line.split()[1].strip("/")
            folder_names.append(folder_name)
    return folder_names


def append_to_file(folders, filename="output.txt"):
    with open(filename, "a") as file:
        for folder in folders:
            file.write(folder + "\n")


async def send_request(semaphore, session, stack, scene_id, coordinates, API_KEY):
    async with semaphore:
        url = f"https://europe-west1-cerulean-338116.cloudfunctions.net/cerulean-cloud-{stack}-cf-sr"
        payload = {
            "Records": [
                {
                    "Sns": {
                        "Message": json.dumps(
                            {
                                "id": scene_id,
                                "mode": scene_id.split("_")[1],
                                "resolution": scene_id.split("_")[2][-1],
                                "polarization": scene_id.split("_")[3][-2:],
                                "footprint": {
                                    "type": "Polygon",
                                    "coordinates": coordinates,
                                },
                            }
                        )
                    }
                }
            ]
        }
        headers = {
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        }
        try:
            await session.post(url, json=payload, headers=headers, timeout=60.0)
        except Exception as e:
            print(f"Error for scene_id: {scene_id}")
            print(e)


print(f"RUNNING ON: {stack}")
while current_date <= end_date:
    prefix = f"GRD/{current_date.year}/{current_date.month}/{current_date.day}/IW/DV/"
    command = f"aws s3 ls s3://{bucket_name}/{prefix} --request-payer requester"
    output = subprocess.check_output(command, shell=True)
    folders = extract_folders(output)
    id_footprint_dict = {}
    for folder in tqdm.tqdm(folders):
        file_key = f"{prefix}{folder}/productInfo.json"  # Ensure the correct path
        try:
            obj = s3_client.get_object(
                Bucket=bucket_name, Key=file_key, RequestPayer="requester"
            )
            content = obj["Body"].read()
            data = json.loads(content)
            # Extract 'id' and 'footprint' and add to dictionary
            # • store to GCP manifest
            id_footprint_dict[data["id"]] = data["footprint"]["coordinates"]
        except s3_client.exceptions.NoSuchKey:
            print(f"File not found: {file_key}")
        except Exception as e:
            print(f"Error accessing file {file_key}: {e}")

    async with httpx.AsyncClient() as session:
        tasks = [
            send_request(semaphore, session, stack, k, v, API_KEY)
            for k, v in id_footprint_dict.items()
        ]
        await asyncio.gather(*tasks)
    print("shipped")

    current_date += datetime.timedelta(days=1)
print("wrapped")